# 3. Dashboard results and explanations

This notebook reads the same saved artifacts as the dashboard. Run notebook 2 to regenerate them; the included artifacts can also be inspected without retraining.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pumpwatch" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "pumpwatch" / "train.py").exists(), "Open from the project root or notebooks folder"
sys.path.insert(0, str(ROOT))
import json
import joblib
import pandas as pd
from pumpwatch.train import ARTIFACTS, metrics
from pumpwatch.inference import explain
from pumpwatch.data import DATA
report = json.loads((ARTIFACTS / "report.json").read_text())
selected = report["selected_model"]
predictions = pd.read_csv(ARTIFACTS / "predictions.csv.gz", parse_dates=["timestamp", "prediction_time"])
bundle = joblib.load(ARTIFACTS / "best_model.joblib")  # trusted model shipped with this project
print("Dashboard selected model:", selected, "| saved bundle:", bundle["name"])
assert selected == bundle["name"]


## Held-out comparison

Precision is the fraction of alerts that match positive windows; recall is the fraction of positive windows alerted. Average precision summarizes the precision–recall ranking. The selected threshold comes from validation.

In [ ]:
comparison = pd.DataFrame({name: item["test"] for name, item in report["models"].items()}).T
cols = ["precision", "recall", "f1", "average_precision", "threshold", "rows", "positive_rows"]
display(comparison[cols])
print("Selected-model test confusion matrix [actual 0/1] × [predicted 0/1]:")
display(pd.DataFrame(report["models"][selected]["test"]["confusion_matrix"],
                     index=["actual 0", "actual 1"], columns=["alert 0", "alert 1"]))


## Check dashboard predictions

The saved predictions include ambiguous windows for the historical explorer. We exclude those windows when checking binary test metrics.

In [ ]:
test = predictions[(predictions.split == "test") & (predictions.target >= 0)].copy()
observed = metrics(test.target, test.risk_score, bundle["threshold"])
for key in ["precision", "recall", "f1", "average_precision", "roc_auc"]:
    assert abs(observed[key] - report["models"][selected]["test"][key]) < 1e-10, key
assert observed["confusion_matrix"] == report["models"][selected]["test"]["confusion_matrix"]
print("Saved predictions reproduce all selected-model test metrics.")


## Event detection and baselines

Event recall asks whether any of the three labeled onset minutes triggered an alert. False alert rates refer only to eligible negative minutes in these collected days.

In [ ]:
display(pd.read_csv(ARTIFACTS / "event_results.csv").head(10))
print(report["event_evaluation"])
display(pd.DataFrame(report["baselines"]).T[["precision", "recall", "f1", "average_precision"]])
print("OHLCV-only test:", {k: report["ohlcv_only_test"][k] for k in ["precision", "recall", "f1"]})


## Feature importance and one example

Permutation importance measures how validation average precision changes when a feature is shuffled. The local chart changes one feature to its training median and measures the score change. Neither proves causation.

In [ ]:
display(pd.read_csv(ARTIFACTS / "importance.csv").head(10))
example = test[test.alert & (test.target == 1)].head(1)
if example.empty:
    example = test.head(1)
# The prediction file contains the engineered features used by the dashboard explorer.
display(example[["timestamp", "symbol", "target", "risk_score", "alert"]])
display(explain(example.iloc[0], bundle).head(10))


**Presentation note:** Our test collection is event-focused and ends in January 2021. The result does not measure performance on all coins or current markets.